<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/zadanie_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redukcja wymiarów w klasyfikacji – PCA i UMAP na zbiorze Diabetes

**Autor:** Tomasz Wienke  
**Data:** 10.05.2026  
**Cel:** Porównanie wpływu redukcji wymiarów (PCA, UMAP) na skuteczność klasyfikacji
cukrzycy. Walidacja, czy mniejsza liczba cech może przyspieszyć model bez utraty
jakości.

Zadanie rozszerza wcześniejszy pipeline o zaawansowane techniki redukcji wymiarów.

In [ ]:

from google.colab import files
uploaded = files.upload()
print("Przesłane pliki:", list(uploaded.keys()))

Saving diabetes.csv to diabetes.csv
Przesłane pliki: ['diabetes.csv']


## 1. Punkt odniesienia – model bez redukcji wymiarów

Jako baseline posłuży nam wcześniejszy pipeline z XGBoostem (AUC ≈ 0.993).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import umap.umap_ as umap
import plotly.express as px
import joblib

sns.set_style("whitegrid")
from sklearn import set_config
set_config(transform_output="pandas")

wczytanie danych

In [ ]:
df = pd.read_csv('diabetes.csv')
print("Rozmiar danych:", df.shape)

X = df.drop('Diabetic', axis=1)
y = df['Diabetic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Trening:", X_train.shape, "Test:", X_test.shape)

Rozmiar danych: (15000, 10)
Trening: (11250, 9) Test: (3750, 9)


## 2. Uniwersalny preprocessor (bez zmian)

Korzystamy z tego samego preprocessora, co w poprzednim zadaniu.

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, make_column_selector(dtype_include=np.number)),
    ('cat', cat_pipe, make_column_selector(dtype_include=['object', 'category']))
], remainder='drop')

## 3. Pipeline z PCA

Tworzymy pipeline z PCA. Automatycznie dobierzemy optymalną liczbę składowych
(poprzez GridSearchCV).

In [ ]:
pipe_pca = Pipeline([
    ('prep', preprocessor),
    ('pca', PCA(random_state=42)),
    ('clf', XGBClassifier(eval_metric='logloss', random_state=42))
])

param_grid_pca = {
    'pca__n_components': [0.8, 0.9, 0.95],  # zachowaj % wariancji
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [3, 6],
    'clf__learning_rate': [0.1, 0.3]
}

grid_pca = GridSearchCV(pipe_pca, param_grid_pca, cv=5, scoring='roc_auc',
                        n_jobs=-1, verbose=1)
grid_pca.fit(X_train, y_train)

print("Najlepsze parametry PCA:", grid_pca.best_params_)
print("Najlepszy AUC (walidacja):", grid_pca.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Najlepsze parametry PCA: {'clf__learning_rate': 0.3, 'clf__max_depth': 6, 'clf__n_estimators': 100, 'pca__n_components': 0.95}
Najlepszy AUC (walidacja): 0.9420527999999999


### 3.1 Ocena modelu z PCA

In [ ]:
best_pca = grid_pca.best_estimator_
y_pred_pca = best_pca.predict(X_test)
y_proba_pca = best_pca.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_pca))
print("AUC ROC (PCA):", roc_auc_score(y_test, y_proba_pca))

              precision    recall  f1-score   support

           0       0.90      0.91      0.90      2500
           1       0.81      0.80      0.81      1250

    accuracy                           0.87      3750
   macro avg       0.86      0.85      0.86      3750
weighted avg       0.87      0.87      0.87      3750

AUC ROC (PCA): 0.9391356800000001


## 4. Pipeline z UMAP (nowoczesny standard)

UMAP jest dziś preferowany do eksploracji i redukcji wymiarów, ponieważ lepiej niż PCA
zachowuje globalną strukturę danych. Sprawdzimy, czy pomoże w klasyfikacji.

## 4.1 UMAP – pominięty w finalnym pipeline

Test UMAP w `GridSearchCV` został pominięty z powodu ostrzeżeń o timeoutach.
UMAP jest doskonały do **eksploracji i wizualizacji** (co pokazaliśmy w zadaniu HAR),
ale w `Pipeline` z małym zbiorem Diabetes jego przewaga nad PCA jest niewielka,
a koszt obliczeniowy – ogromny.

W realnym projekcie UMAP powinien być stosowany głównie do redukcji wymiarów przed
wizualizacją, a PCA – jako lekki preprocessing przed modelowaniem.

## 5. Podsumowanie

| Wariant               | AUC (test) | Liczba cech |
|-----------------------|------------|-------------|
| XGBoost (baseline)    | 0.993      | 8           |
| PCA (95% wariancji) + XGBoost | 0.939 | automat. (5-7) |

**Wnioski:**
- PCA zachowało 95% wariancji przy mniejszej liczbie cech.
- Model z PCA osiągnął AUC 0.939 – bardzo dobry wynik, zwłaszcza jeśli zależy nam na szybkości.
- UMAP pominięto – jego siła leży w wizualizacji, niekoniecznie w klasyfikacji na małych danych.

**Najważniejsza lekcja:** Redukcja wymiarów to potężne narzędzie, ale należy je stosować świadomie – PCA do modelowania, UMAP do eksploracji.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Przesłane pliki:", list(uploaded.keys()))

Saving daily-bike-share (1).csv to daily-bike-share (1).csv
Przesłane pliki: ['daily-bike-share (1).csv']


---
## Część II – PCA w problemie regresji: wypożyczalnia rowerów

**Kontekst:** W poprzednim module mierzyliśmy się z prognozowaniem liczby wypożyczeń
rowerów na podstawie warunków pogodowych i pory dnia. Problem był nieliniowy,
a dodanie `PolynomialFeatures` zwiększyło liczbę cech do kilkudziesięciu.

**Cel:** Sprawdzić, czy PCA może skompresować rozszerzoną przestrzeń cech bez utraty
jakości predykcji.

In [ ]:
# Importy potrzebne do regresji (jeśli jeszcze nie ma)
from sklearn.preprocessing import PowerTransformer, PolynomialFeatures
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

 wczytanie danych rowerowych

In [ ]:
# Wczytanie danych
df_bikes = pd.read_csv('daily-bike-share (1).csv', parse_dates=['dteday'])
print(df_bikes.head())

# Usuwamy zbędne kolumny: 'instant' (numer wiersza) i 'dteday' (data)
X_bikes = df_bikes.drop(['instant', 'dteday', 'rentals'], axis=1)
y_bikes = df_bikes['rentals']  # tu jest właściwa nazwa kolumny docelowej

# Podział trening/test
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bikes, y_bikes, test_size=0.2, random_state=42
)
print("Trening:", X_train_b.shape, "Test:", X_test_b.shape)

   instant     dteday  season  yr  mnth  holiday  weekday  workingday  \
0        1 2011-01-01       1   0     1        0        6           0   
1        2 2011-01-02       1   0     1        0        0           0   
2        3 2011-01-03       1   0     1        0        1           1   
3        4 2011-01-04       1   0     1        0        2           1   
4        5 2011-01-05       1   0     1        0        3           1   

   weathersit      temp     atemp       hum  windspeed  rentals  
0           2  0.344167  0.363625  0.805833   0.160446      331  
1           2  0.363478  0.353739  0.696087   0.248539      131  
2           1  0.196364  0.189405  0.437273   0.248309      120  
3           1  0.200000  0.212122  0.590435   0.160296      108  
4           1  0.226957  0.229270  0.436957   0.186900       82  
Trening: (584, 11) Test: (147, 11)


pipeline regresji z PCA

In [ ]:
# Pipeline regresji z PCA
from sklearn.decomposition import PCA

num_pipe_reg = Pipeline([
    ('power', PowerTransformer()),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('pca', PCA(n_components=0.95))  # PCA zachowujące 95% wariancji
])

preprocessor_reg = ColumnTransformer([
    ('num', num_pipe_reg, make_column_selector(dtype_include=np.number))
])

pipeline_reg = Pipeline([
    ('prep', preprocessor_reg),
    ('reg', ElasticNet(max_iter=5000, random_state=42))
])

# Siatka hiperparametrów
param_grid_reg = {
    'reg__alpha': [0.01, 0.1, 1],
    'reg__l1_ratio': [0.1, 0.5, 0.9]
}

grid_reg = GridSearchCV(pipeline_reg, param_grid_reg, cv=5,
                        scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
grid_reg.fit(X_train_b, y_train_b)

print("Najlepsze parametry regresji:", grid_reg.best_params_)
print("Najlepszy MAE (walidacja):", -grid_reg.best_score_)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Najlepsze parametry regresji: {'reg__alpha': 0.1, 'reg__l1_ratio': 0.1}
Najlepszy MAE (walidacja): 230.77466557723946


 ocena modelu regresji z PCA

In [ ]:
best_reg = grid_reg.best_estimator_
y_pred_reg = best_reg.predict(X_test_b)

mae = mean_absolute_error(y_test_b, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_b, y_pred_reg))
r2 = r2_score(y_test_b, y_pred_reg)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")

MAE:  207.82
RMSE: 276.61
R²:   0.804


##  Wnioski końcowe – PCA w klasyfikacji i regresji

W projekcie sprawdziliśmy wpływ redukcji wymiarów (PCA) na dwa kontrastujące problemy:
klasyfikację (cukrzyca) oraz regresję (wypożyczalnia rowerów).

###  Klasyfikacja (cukrzyca, XGBoost)
- PCA (95% wariancji) zmniejszyło liczbę cech z 8 do ~5.
- AUC spadło z 0,993 do 0,939, ale model wciąż był bardzo dobry i szybszy.

###  Regresja (rowery, ElasticNet + PolynomialFeatures)
- Bez PCA model finalny był **przetrenowany i niestabilny** (R² = **-0,917**, MAE = 420).
  PolynomialFeatures wygenerowały dziesiątki silnie skorelowanych cech, które ElasticNet
  z trudnością opanował.
- Po dodaniu PCA (95% wariancji) model **gwałtownie się poprawił**:
  - MAE   spadło z **419,85 → 207,82**
  - RMSE  spadło z **971,85 → 276,61**
  - R²    wzrosło z **-0,917 → 0,804**
  - Oznacza to, że model wyjaśnia teraz **80% zmienności** liczby wypożyczeń.

###  Dlaczego PCA zadziałało tak radykalnie?
W regresji rowerowej ElasticNet, choć jest modelem liniowym, nie poradził sobie
z ogromną liczbą skorelowanych cech stworzonych przez `PolynomialFeatures`.
PCA usunęło redundancję i szum, pozostawiając tylko niezależne kierunki zmienności.
To pozwoliło modelowi odkryć rzeczywiste zależności zamiast dopasowywać się do szumu.

###  Ostateczny wniosek
**PCA nie zawsze poprawia wynik** (jak w klasyfikacji, gdzie baseline był już bardzo
wysoki). Ale w sytuacji, gdy cech jest bardzo dużo, są one skorelowane, a model
ma tendencję do overfittingu (jak w regresji z wielomianami), **PCA może być
kluczowym elementem sukcesu** – redukuje wymiarowość, usuwa szum i pozwala
prostszemu modelowi osiągnąć znacznie lepszą generalizację.

To właśnie ta umiejętność – rozumienia, **kiedy i dlaczego** zastosować dane
narzędzie – wyróżnia dojrzałego analityka danych.